# Llama-3.1-8B-Instruct — AEGIS Text-to-SQL Fine-Tuning

Fine-tunes **`meta-llama/Llama-3.1-8B-Instruct`** with QLoRA on the merged BIRD+Spider corpus (1,000 examples),
evaluates a quick proxy on the held-out test split, and pushes the result to
**`Daveonyango254/Llama31-8B-Aegis-Text2SQL`**.

**Why this model.** Llama-3.1-8B-Instruct is the continuity baseline: the predecessor agentic system's local generator was Llama-3.1-8B, so this run answers 'what does the same family gain from domain SFT?' directly. It brings a 128K context (headroom for very wide enterprise schemas), the largest tooling ecosystem of any open model, and strong general reasoning. It is not expected to beat the Coder-7B on raw EX at equal size, and that comparison is exactly the point. Note: the repo is license-gated — accept the license on the model page with the same HF account as your token before running.

**Run non-interactively** (recommended on RunPod, inside tmux):
```bash
papermill notebooks/llama31_aegis.ipynb outputs/llama31_aegis_output.ipynb -f configs/llama31.yaml
```
All knobs below are papermill parameters; see `DETAILED_GUIDE.md` for what each one does,
why this value was chosen, and how changing it moves accuracy/cost.

In [ ]:
# Parameters (papermill overrides land here)
data_dir = 'data'
train_file = 'train.jsonl'
val_file = 'validation.jsonl'
test_file = 'test.jsonl'
max_seq_len = 4096
epochs = 3
warmup_ratio = 0.03
weight_decay = 0.0
lora_dropout = 0.05
seed = 42
output_root = 'outputs'
push_to_hub = True
merge_adapter = False
resume = True
quick_eval_n = 50
logging_steps = 10
model_id = 'meta-llama/Llama-3.1-8B-Instruct'
hub_repo = 'Daveonyango254/Llama31-8B-Aegis-Text2SQL'
response_template = '<|start_header_id|>assistant<|end_header_id|>\n\n'
target_modules = ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj']
lora_r = 32
lora_alpha = 64
learning_rate = 0.0001
per_device_bs = 2
grad_accum = 8
run_name = 'llama31_aegis'

In [ ]:
# 1) Environment ----------------------------------------------------------
# RunPod PyTorch images ship torch+CUDA. Everything else is pinned in requirements.txt:
#   pip install -r requirements.txt
import os, sys, json, time, random, re
import torch
from dotenv import load_dotenv

load_dotenv()                                   # reads HF_TOKEN from .env
HF_TOKEN = os.environ.get("HF_TOKEN")
assert HF_TOKEN, "HF_TOKEN not found — copy .env.example to .env and fill it in"
from huggingface_hub import login
login(token=HF_TOKEN)

assert torch.cuda.is_available(), "No CUDA device visible — check RunPod GPU + drivers"
print("GPU:", torch.cuda.get_device_name(0),
      f"| VRAM {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
import transformers, peft, trl, bitsandbytes, datasets as hf_datasets
print("transformers", transformers.__version__, "| trl", trl.__version__,
      "| peft", peft.__version__, "| bitsandbytes", bitsandbytes.__version__)

random.seed(seed); torch.manual_seed(seed)
os.makedirs(f"{output_root}/{run_name}", exist_ok=True)

In [ ]:
# 2) Data -----------------------------------------------------------------
# Records are chat triples {system,user,assistant}; we render each with the model's own
# chat template into a single "text" field. Loss is masked to the assistant span only
# (completion-only) so the model is never trained to regenerate schemas.
from datasets import load_dataset

data_files = {"train": f"{data_dir}/{train_file}", "validation": f"{data_dir}/{val_file}"}
ds = load_dataset("json", data_files=data_files)
print(ds)

from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token   # required for batching (Llama, Phi)

def render(ex):
    return {"text": tokenizer.apply_chat_template(ex["messages"], tokenize=False,
                                                  add_generation_prompt=False)}
ds = ds.map(render, remove_columns=["messages"])
print(ds["train"][0]["text"][:600], "…")

lens = [len(tokenizer(t).input_ids) for t in ds["train"]["text"][:200]]
print(f"token lengths (first 200): p50={sorted(lens)[100]} max={max(lens)} "
      f"(max_seq_len={max_seq_len} — raise it if max approaches the cap)")

In [ ]:
# 3) Model — 4-bit NF4 QLoRA base ------------------------------------------
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

bnb = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True)

attn_impl = "sdpa"
try:                                             # flash-attn is optional; sdpa is a safe fallback
    import flash_attn                            # noqa: F401
    attn_impl = "flash_attention_2"
except ImportError:
    pass
print("attention implementation:", attn_impl)

model = AutoModelForCausalLM.from_pretrained(
    model_id, quantization_config=bnb, torch_dtype=torch.bfloat16,
    attn_implementation=attn_impl, device_map="auto", trust_remote_code=True)
model.config.use_cache = False                   # incompatible with gradient checkpointing

In [ ]:
# 4) LoRA ------------------------------------------------------------------
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)
lora = LoraConfig(r=lora_r, lora_alpha=lora_alpha, lora_dropout=lora_dropout,
                  bias="none", task_type="CAUSAL_LM", target_modules=target_modules)
model = get_peft_model(model, lora)
model.print_trainable_parameters()

In [ ]:
# 5) Trainer ---------------------------------------------------------------
from trl import SFTTrainer, SFTConfig, DataCollatorForCompletionOnlyLM

collator = DataCollatorForCompletionOnlyLM(response_template=response_template,
                                           tokenizer=tokenizer)
args = SFTConfig(
    output_dir=f"{output_root}/{run_name}", run_name=run_name,
    num_train_epochs=epochs, learning_rate=learning_rate,
    per_device_train_batch_size=per_device_bs, gradient_accumulation_steps=grad_accum,
    per_device_eval_batch_size=per_device_bs,
    lr_scheduler_type="cosine", warmup_ratio=warmup_ratio, weight_decay=weight_decay,
    bf16=True, gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    max_seq_length=max_seq_len, dataset_text_field="text", packing=False,
    logging_steps=logging_steps, eval_strategy="epoch",
    save_strategy="epoch", save_total_limit=2,
    load_best_model_at_end=True, metric_for_best_model="eval_loss",
    seed=seed, report_to="none")

trainer = SFTTrainer(model=model, args=args, processing_class=tokenizer,
                     train_dataset=ds["train"], eval_dataset=ds["validation"],
                     data_collator=collator)

In [ ]:
# 6) Train (auto-resumes from the last checkpoint if one exists) -----------
import glob
last_ckpt = None
if resume:
    ckpts = sorted(glob.glob(f"{output_root}/{run_name}/checkpoint-*"),
                   key=lambda p: int(p.rsplit("-", 1)[1]))
    last_ckpt = ckpts[-1] if ckpts else None
    print("resuming from:", last_ckpt or "scratch")
t0 = time.time()
trainer.train(resume_from_checkpoint=last_ckpt)
print(f"training wall-clock: {(time.time()-t0)/60:.1f} min")
print(trainer.state.log_history[-3:])

In [ ]:
# 7) Quick proxy evaluation on the test split ------------------------------
# sqlglot parse rate (floor sanity) + normalized exact match (conservative EX proxy).
# Full BIRD Execution Accuracy needs the BIRD databases + official evaluator — see
# DETAILED_GUIDE.md "Full BIRD evaluation".
import sqlglot
test = [json.loads(l) for l in open(f"{data_dir}/{test_file}")][:quick_eval_n]

def norm(s):
    s = re.sub(r"\s+", " ", s.strip().rstrip(";")).strip()
    return re.sub(r"\s*([(),=<>])\s*", r"\1", s).upper()

model.eval(); model.config.use_cache = True
parse_ok = em = 0; samples = []
for i, rec in enumerate(test):
    prompt = tokenizer.apply_chat_template(rec["messages"][:2], tokenize=False,
                                           add_generation_prompt=True)
    ids = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**ids, max_new_tokens=256, do_sample=False,
                             pad_token_id=tokenizer.pad_token_id)
    sql = tokenizer.decode(out[0][ids.input_ids.shape[1]:], skip_special_tokens=True).strip()
    sql = sql.split("```")[-2].replace("sql", "", 1).strip() if "```" in sql else sql
    gold = rec["messages"][2]["content"]
    try:
        sqlglot.parse_one(re.sub(r"`[^`]*`", "x", sql), read="sqlite"); parse_ok += 1
    except Exception:
        pass
    em += norm(sql) == norm(gold)
    if i < 3: samples.append((rec["messages"][1]["content"].split("Question:")[-1][:120], sql, gold))

n = len(test)
metrics = {"n": n, "parse_rate": round(parse_ok/n, 4), "normalized_em": round(em/n, 4)}
print(metrics)
json.dump(metrics, open(f"{output_root}/{run_name}/quick_eval.json", "w"), indent=1)
for q, p, g in samples:
    print("\nQ:", q.strip(), "\nPRED:", p, "\nGOLD:", g)
model.config.use_cache = False

In [ ]:
# 8) Save adapter (and optionally a merged fp16 model) ----------------------
adapter_dir = f"{output_root}/{run_name}/adapter"
trainer.save_model(adapter_dir)
tokenizer.save_pretrained(adapter_dir)
json.dump({k: v for k, v in vars().items()
           if k in ("model_id","learning_rate","epochs","lora_r","lora_alpha",
                    "lora_dropout","target_modules","per_device_bs","grad_accum",
                    "max_seq_len","warmup_ratio","seed")},
          open(f"{adapter_dir}/training_config.json", "w"), indent=1)
print("adapter saved:", adapter_dir)

if merge_adapter:                                # fp16 merged weights for vLLM serving
    merged = trainer.model.merge_and_unload()
    merged_dir = f"{output_root}/{run_name}/merged"
    merged.save_pretrained(merged_dir, safe_serialization=True)
    tokenizer.save_pretrained(merged_dir)
    print("merged model saved:", merged_dir)

In [ ]:
# 9) Model card + push to the Hub ------------------------------------------
card = f"""---
license: other
base_model: {model_id}
tags: [text-to-sql, bird, spider, qlora, aegis-sql]
---
# {hub_repo.split('/')[-1]}

QLoRA fine-tune of `{model_id}` for Text-to-SQL, trained as the local-path
generator of the AEGIS-SQL hybrid system (three-axis constrained NL2SQL:
accuracy / cost / privacy).

- **Data**: 1,000 curated examples — 500 BIRD-train + 500 diversity-stratified
  Spider-train (set ops, HAVING, nested, multi-join rebalanced); splits 798/101/101.
- **Recipe**: 4-bit NF4 QLoRA, r={lora_r} alpha={lora_alpha}, lr={learning_rate},
  {epochs} epochs, completion-only loss, max_seq_len={max_seq_len}, seed={seed}.
- **Prompt format**: chat messages; user = `Database:` + schema (Tables + Foreign
  Keys) + `Question:` + `Return only SQL.`; assistant = bare SQL.
- **Quick proxy on held-out test**: see `quick_eval.json` in this repo.
  Full BIRD EX requires the official evaluator + databases.

Trained with the text2sql_aegis_experiment package (notebooks/llama31_aegis.ipynb).
"""
if push_to_hub:
    from huggingface_hub import HfApi
    open(f"{adapter_dir}/README.md", "w").write(card)
    api = HfApi(token=HF_TOKEN)
    api.create_repo(hub_repo, exist_ok=True)
    api.upload_folder(folder_path=adapter_dir, repo_id=hub_repo)
    api.upload_file(path_or_fileobj=f"{output_root}/{run_name}/quick_eval.json",
                    path_in_repo="quick_eval.json", repo_id=hub_repo)
    print("pushed:", f"https://huggingface.co/{hub_repo}")
else:
    print("push_to_hub=False — skipped")

In [ ]:
# 10) Inference example ----------------------------------------------------
def generate_sql(question: str, db_id: str, schema_text: str,
                 max_new_tokens: int = 256) -> str:
    """schema_text: the 'Tables ... Foreign Keys ...' block exactly as in training."""
    user = (f"Database: {db_id}\n\nSchema:\n\n{schema_text}\n\n"
            f"Question:\n{question}\n\nReturn only SQL.")
    msgs = [{"role": "system", "content":
             "You are an expert SQL generator. Return only executable SQL with no explanation."},
            {"role": "user", "content": user}]
    prompt = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    ids = tokenizer(prompt, return_tensors="pt").to(model.device)
    model.config.use_cache = True
    with torch.no_grad():
        out = model.generate(**ids, max_new_tokens=max_new_tokens, do_sample=False,
                             pad_token_id=tokenizer.pad_token_id)
    model.config.use_cache = False
    return tokenizer.decode(out[0][ids.input_ids.shape[1]:], skip_special_tokens=True).strip()

demo_schema = "Tables\n\nemployees(\n    id,\n    name,\n    hired\n)"
print(generate_sql("List names of employees hired after 2020.", "demo_db", demo_schema))

## Plugging the model into the AEGIS agentic system

The AEGIS local path calls one function: `generate_sql(question, schema_text, evidence)`.
The adapter below is a drop-in `LocalGenerator` for the graph pipeline (Query Planner →
router → **local SLM** → Reviewer). Candidate sampling feeds the pipeline's
execution-guided self-consistency lever — the Reviewer executes the candidates and votes
over result sets, so `n_candidates > 1` is where the harness earns its accuracy.

In [ ]:
# 11) AEGIS local-path adapter ---------------------------------------------
class LocalGenerator:
    """Drop-in local SLM for the AEGIS graph pipeline.

    Contract (matches the Query Planner Agent's output):
      generate(question, db_id, schema_text, evidence=None, n_candidates=4)
        -> list[str] SQL candidates (greedy first, then temperature samples),
      which the Reviewer executes and votes over (result-set self-consistency).
    """
    def __init__(self, model, tokenizer, system_prompt=None):
        self.m, self.t = model, tokenizer
        self.sys = system_prompt or ("You are an expert SQL generator. "
                                     "Return only executable SQL with no explanation.")

    def _prompt(self, question, db_id, schema_text, evidence):
        ev = f"\n\nEvidence:\n{evidence}" if evidence else ""
        user = (f"Database: {db_id}\n\nSchema:\n\n{schema_text}{ev}\n\n"
                f"Question:\n{question}\n\nReturn only SQL.")
        msgs = [{"role": "system", "content": self.sys},
                {"role": "user", "content": user}]
        return self.t.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

    def generate(self, question, db_id, schema_text, evidence=None, n_candidates=4):
        ids = self.t(self._prompt(question, db_id, schema_text, evidence),
                     return_tensors="pt").to(self.m.device)
        outs, self.m.config.use_cache = [], True
        with torch.no_grad():
            g = self.m.generate(**ids, max_new_tokens=256, do_sample=False,
                                pad_token_id=self.t.pad_token_id)
            outs.append(g[0][ids.input_ids.shape[1]:])
            if n_candidates > 1:
                s = self.m.generate(**ids, max_new_tokens=256, do_sample=True,
                                    temperature=0.8, top_p=0.95,
                                    num_return_sequences=n_candidates - 1,
                                    pad_token_id=self.t.pad_token_id)
                outs += [row[ids.input_ids.shape[1]:] for row in s]
        self.m.config.use_cache = False
        return [self.t.decode(o, skip_special_tokens=True).strip() for o in outs]

gen = LocalGenerator(model, tokenizer)
print(gen.generate("How many employees were hired after 2020?", "demo_db",
                   demo_schema, n_candidates=2))

### Serving for the agent at scale (vLLM)

For pipeline-scale evaluation, serve the **merged** model (set `merge_adapter = True`,
re-run cell 8) behind an OpenAI-compatible endpoint and point the AEGIS local path at it:

```bash
python -m vllm.entrypoints.openai.api_server \
    --model outputs/llama31_aegis/merged \
    --served-model-name llama31_aegis \
    --max-model-len 4096 --gpu-memory-utilization 0.90
```

Troubleshooting (OOM, HF auth, bitsandbytes, flash-attn, resume): see
`DETAILED_GUIDE.md` — every failure mode listed there includes its exact fix.